# Indicator Template

Compute custom indicators from order book data and export as CSV for the Prosperity Visualizer.

**Output format**: comma-delimited CSV with `timestamp` as the first column and indicator values as subsequent columns.

```
timestamp,WallMid,FairValue
0,10001.5,10002.0
100,10003.0,10003.5
```

In [ ]:
import pandas as pd
import numpy as np

PRODUCT = 'ASH_COATED_OSMIUM'
DAY = 0

prices = pd.read_csv(f'../ROUND1/prices_round_1_day_{DAY}.csv', sep=';')
prices = prices[prices['product'] == PRODUCT].copy()
prices.head()

In [ ]:
def volume_weighted_mid(row):
    """Compute volume-weighted mid price from best bid/ask."""
    bp, bv = row.get('bid_price_1'), row.get('bid_volume_1')
    ap, av = row.get('ask_price_1'), row.get('ask_volume_1')
    if pd.isna(bp) and pd.isna(ap):
        return np.nan
    if pd.isna(bp):
        return ap
    if pd.isna(ap):
        return bp
    total_vol = bv + av
    if total_vol == 0:
        return (bp + ap) / 2
    return (bp * av + ap * bv) / total_vol

prices['WallMid'] = prices.apply(volume_weighted_mid, axis=1)
prices[['timestamp', 'mid_price', 'WallMid']].head(10)

In [ ]:
# Add your own indicators here
# Example: exponential moving average of mid price
prices['EMA_20'] = prices['mid_price'].ewm(span=20).mean()

prices[['timestamp', 'WallMid', 'EMA_20']].head(10)

In [ ]:
output = prices[['timestamp', 'WallMid', 'EMA_20']].dropna()
output.to_csv(f'indicators_{PRODUCT}_day{DAY}.csv', index=False)
print(f'Exported {len(output)} rows to indicators_{PRODUCT}_day{DAY}.csv')